In [ ]:
import pandas as pd

## 1. Load Results

Load the pipeline results from the Excel export and the raw dataset CSVs.


# Error Analysis & Investigation


---

This notebook investigates the false positive (FP) cases from the product matching pipeline results. The goal is to understand **why** the model makes incorrect predictions and identify areas for improvement.


## Key Findings / Temel Bulgular

- **FP (wrong match):** The model retrieves the correct product but matches it to a *similar but different* variant (e.g., different pack size, color, or carrier).
- **FP (no match in DB):** The model predicts a match for queries that have no corresponding product in the database. Visual inspection shows these retrieved products are often *extremely similar* to the query.
- **Dataset quality:** Some ground truth annotations appear incorrect — the model's predictions are sometimes more accurate than the labeled ground truth.

---

In [ ]:
results = pd.read_excel("product_matching_results.xlsx", sheet_name="Detailed Results")
results.head()

In [ ]:
results.loc[results["Result"] == "FP (wrong match)"]

In [ ]:
products = pd.read_csv("dataset/products.csv")
products.head()

In [ ]:
queries = pd.read_csv("dataset/queries.csv")
queries.head()

## 2. FP (Wrong Match) Analysis 

These are cases where the query **has a match** in the database, but the model retrieved a **different product** (not the ground truth). We visualize the query, the retrieved (wrong) match, and the actual ground truth side-by-side.



In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# FP (wrong match) - show query, retrieved, and ground truth images side-by-side
fp = results[results["Result"] == "FP (wrong match)"]

for idx, row in fp.iterrows():
    print(f"\n=== {row['Query ID']} ===")
    print(f"Query: {row['Query Name']}")
    print(f"Matched to: {row['Top-1 Match Name']}")
    print(f"Should match: {row['Ground Truth Name']}")
    
    # Get image paths using .values[0] to extract single value
    query_img = queries.loc[queries["query_id"] == row["Query ID"], "image"].values[0]
    retrieved_img = products.loc[products["db_id"] == row["Top-1 Match ID"], "image"].values[0]
    gt_img = products.loc[products["db_id"] == row["Ground Truth ID"], "image"].values[0]
    
    # Display images side by side
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(Image.open(f"dataset/images/{query_img}"))
    axes[0].set_title("Query Image")
    axes[0].axis('off')
    
    axes[1].imshow(Image.open(f"dataset/images/{retrieved_img}"))
    axes[1].set_title("Retrieved (Wrong Match)")
    axes[1].axis('off')
    
    axes[2].imshow(Image.open(f"dataset/images/{gt_img}"))
    axes[2].set_title("Ground Truth")
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

### Observation

**EN:** In general, the model's selected matches are more accurate than the ground truth labels. The dataset has quality issues — some ground truth annotations are questionable. For example, the model correctly identifies the same product with a different seller/condition (e.g., "Restored" vs "New"), but the ground truth points to a different variant entirely.



## 3. FP (No Match in DB) Analysis

These are cases where the query has **no match** in the database (`has_match=False`), but the model still predicted a match. We examine whether the retrieved products are visually and semantically similar to the query.



In [ ]:
results.loc[results["Result"] == "FP"]

In [ ]:
# FP (no match in DB) - show query and retrieved image side-by-side
fp_nomatch = results[results["Result"] == "FP"]

for idx, row in fp_nomatch.iterrows():
    print(f"\n=== {row['Query ID']} ===")
    print(f"Query: {row['Query Name']}")
    print(f"Matched to: {row['Top-1 Match Name']}")
    print(f"Should match: {row['Ground Truth Name']}")

    # Get image paths
    query_match = queries.loc[queries["query_id"] == row["Query ID"], "image"]
    retrieved_match = products.loc[products["db_id"] == row["Top-1 Match ID"], "image"]

    # Skip gracefully if an image record is missing
    if query_match.empty or retrieved_match.empty:
        print("Image path not found; skipping visualization.")
        continue

    query_img = query_match.iloc[0]
    retrieved_img = retrieved_match.iloc[0]

    # Display query and retrieved image side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].imshow(Image.open(f"dataset/images/{query_img}"))
    axes[0].set_title("Query Image")
    axes[0].axis("off")

    axes[1].imshow(Image.open(f"dataset/images/{retrieved_img}"))
    axes[1].set_title("Retrieved Match")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
